# Cached multimodal inversion

This notebook keeps observations at their **native frequencies**. The model is cached on separate uniform grids for FG, P and SH, then interpolated to each observation frequency during the likelihood calculation.

In [1]:
import numpy as np

from das_tools.hammer_inversion import (
    load_events, ParameterGrid,
    set_wave_sigmas, get_fg_cache, invert_cached,
    save_inversion_results,
)
from das_tools.utils import proj_path

# Inputs and outputs live under outputs/tap_inv_batch/ in the project
# (das_tools.utils.proj_path; DAS_LFI_OUTPUT_DIR moves it).
OUT = proj_path.output_dir("tap_inv_batch")
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
MODEL_CONSTANTS = dict(h_w=10.0, rho_i=910.0, rho_w=1025.0, P=0.0)

bounds = [(0.5e9, 10.0e9), (0.05, 0.45), (0.75, 1.25)]
grid = ParameterGrid.from_bounds(
    bounds[0], bounds[1], bounds[2],
    n_E=181,      # ΔE = 0.05 GPa
    n_nu=81,      # Δν = 0.005
    n_h_i=101,    # Δh_i = 0.005 m
)

# FG has no closed form, so it is precomputed on this 1 Hz grid over the FG picks
# (5-50 Hz) and interpolated; P and SH are evaluated exactly at the picks.
FG_FREQUENCY = np.arange(5.0, 50.0 + 0.5, 1.0)

In [3]:
# Events without outlier picks, written by step2_residual_diagnostics.ipynb
events_raw = load_events(OUT / "dispersion_events_0908_clean.npz")

Loaded 23 events from:
/Users/junsuj/Desktop/git/das-landfast-ice-inversion2025/outputs/tap_inv_batch/dispersion_events_0908_clean.npz


In [4]:
# SIGMA_BY_WAVE = {"P": 0.1, "SH": 0.1, "FG": 0.1}
SIGMA_BY_WAVE = {"P": 0.0124*np.sqrt(2.83), 
                 "SH": 0.0292*np.sqrt(4.68), 
                 "FG": 0.0168*np.sqrt(7.84)}
print("\nSIGMA_BY_WAVE =", SIGMA_BY_WAVE)
events = set_wave_sigmas(events_raw, SIGMA_BY_WAVE)


SIGMA_BY_WAVE = {'P': np.float64(0.020860028763163296), 'SH': np.float64(0.06316925834612909), 'FG': np.float64(0.04703999999999999)}


In [5]:
CACHE_FILE = OUT / "dispersion_cache_wavegrids_detailed.npz"

# Reused only if it was built for this grid, MODEL_CONSTANTS and FG_FREQUENCY;
# otherwise rebuilt and saved. rebuild=True forces a rebuild.
cache = get_fg_cache(CACHE_FILE, grid, MODEL_CONSTANTS, FG_FREQUENCY)

In [7]:
event_results, global_result = invert_cached(events, grid, cache)
save_inversion_results(event_results, global_result, OUT / "inversion_results_fixed_0908_customSTD_noMod.npz")